In [1]:
! pip install optuna
! pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

# BigMartSales

Imports and Global Configs

In [16]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

import optuna
import mlflow
import mlflow.sklearn

RANDOM_STATE = 42
N_SPLITS = 5

mlflow.set_experiment("BigMartSales_Prediction")


<Experiment: artifact_location='/content/mlruns/1', creation_time=1770887578297, experiment_id='1', last_update_time=1770887578297, lifecycle_stage='active', name='BigMartSales_Prediction', tags={}>

Data Loading

In [21]:
train = pd.read_csv("/content/train_v9rqX0R.csv")
test = pd.read_csv("/content/test_AbJTz2l.csv")

test_ids = test[["Item_Identifier", "Outlet_Identifier"]].copy()

print(train.shape, test.shape)

(8523, 12) (5681, 11)


Feature Engineering Pipeline

In [22]:
def feature_engineering(train, test):

    train = train.copy()
    test = test.copy()

    # Handling Missing Item Weight
    item_weight_median = train.groupby("Item_Type")["Item_Weight"].median()

    for df in [train, test]:
        df["Item_Weight"] = df.apply(
            lambda x: item_weight_median[x["Item_Type"]]
            if pd.isnull(x["Item_Weight"]) else x["Item_Weight"],
            axis=1
        )

    # Visibility Handling

    train.loc[train["Item_Visibility"] == 0, "Item_Visibility"] = np.nan
    test.loc[test["Item_Visibility"] == 0, "Item_Visibility"] = np.nan

    visibility_median = train.groupby("Item_Type")["Item_Visibility"].median()

    for df in [train, test]:
        df["Item_Visibility"] = df.apply(
            lambda x: visibility_median[x["Item_Type"]]
            if pd.isnull(x["Item_Visibility"]) else x["Item_Visibility"],
            axis=1
        )

    # Fat Content Normalize + Binary Encoding

    mapping = {
        "lf": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular",
        "regular": "Regular"
    }

    for df in [train, test]:
        df["Item_Fat_Content"] = (
            df["Item_Fat_Content"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace(mapping)
        )

    fat_encoding = {"Low Fat": 1, "Regular": 0}

    for df in [train, test]:
        df["Item_Fat_Content"] = df["Item_Fat_Content"].replace(fat_encoding)

    # Outlet Age Generation

    CURRENT_YEAR = 2013
    for df in [train, test]:
        df["Outlet_Age"] = CURRENT_YEAR - df["Outlet_Establishment_Year"]

    train.drop(columns=["Outlet_Establishment_Year"], inplace=True)
    test.drop(columns=["Outlet_Establishment_Year"], inplace=True)


    # Visibility Ratio

    visibility_mean = train.groupby("Item_Type")["Item_Visibility"].transform("mean")
    train["Visibility_Ratio"] = train["Item_Visibility"] / visibility_mean

    visibility_map = train.groupby("Item_Type")["Item_Visibility"].mean()
    test["Visibility_Ratio"] = test["Item_Visibility"] / test["Item_Type"].map(visibility_map)


    # MRP Binning
    bins = [0, 70, 140, 210, 300]

    train["MRP_Bin"] = pd.cut(train["Item_MRP"], bins=bins, labels=False)
    test["MRP_Bin"] = pd.cut(test["Item_MRP"], bins=bins, labels=False)

    # Interaction Feature

    train["Item_Outlet_Type"] = train["Item_Type"] + "_" + train["Outlet_Type"]
    test["Item_Outlet_Type"] = test["Item_Type"] + "_" + test["Outlet_Type"]

    return train, test


Kfold Encoding

In [23]:
def kfold_target_encoding(train_df, test_df, column, target, n_splits=5):

    train_encoded = pd.Series(index=train_df.index, dtype=float)
    global_mean = train_df[target].mean()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for train_idx, val_idx in kf.split(train_df):
        fold_train = train_df.iloc[train_idx]
        fold_val = train_df.iloc[val_idx]

        means = fold_train.groupby(column)[target].mean()
        train_encoded.iloc[val_idx] = fold_val[column].map(means)

    train_encoded.fillna(global_mean, inplace=True)

    full_means = train_df.groupby(column)[target].mean()
    test_encoded = test_df[column].map(full_means)
    test_encoded.fillna(global_mean, inplace=True)

    return train_encoded, test_encoded


Prepare Modeling Data

In [24]:
def build_dataset(train, test):

    train, test = feature_engineering(train, test)

    # Target encoding
    train["Item_TE"], test["Item_TE"] = kfold_target_encoding(
        train, test, "Item_Identifier", "Item_Outlet_Sales"
    )

    train["Outlet_TE"], test["Outlet_TE"] = kfold_target_encoding(
        train, test, "Outlet_Identifier", "Item_Outlet_Sales"
    )

    # Drop high-card columns
    train.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)
    test.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)

    # One hot encode remaining categoricals
    train = pd.get_dummies(train, drop_first=True)
    test = pd.get_dummies(test, drop_first=True)

    train, test = train.align(test, join="left", axis=1, fill_value=0)

    y = train["Item_Outlet_Sales"]
    X = train.drop("Item_Outlet_Sales", axis=1)

    return X, y, test


In [25]:
X, y, test_processed = build_dataset(train, test)


Model Evaluation Utility

In [26]:
def evaluate_model(model, X, y):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        scores.append(rmse)

    return np.mean(scores)


Baseline MLflow Run

In [27]:
with mlflow.start_run(run_name="GB_TargetEncoding_Baseline"):

    model = GradientBoostingRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=RANDOM_STATE
    )

    cv_rmse = evaluate_model(model, X, y)

    mlflow.log_params(model.get_params())
    mlflow.log_metric("cv_rmse", cv_rmse)

    print("Baseline CV RMSE:", cv_rmse)


Baseline CV RMSE: 1095.3109609834323


Optuna + MLflow Integration

In [11]:
def objective(trial):

    with mlflow.start_run(nested=True):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "random_state": RANDOM_STATE
        }

        model = GradientBoostingRegressor(**params)

        cv_rmse = evaluate_model(model, X, y)

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", cv_rmse)

        return cv_rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best CV:", study.best_value)
print("Best Params:", study.best_params)


[I 2026-02-12 09:37:28,087] A new study created in memory with name: no-name-b136b081-83a5-483a-a9f4-c85951fc5568
[I 2026-02-12 09:38:24,070] Trial 0 finished with value: 1084.2619630578715 and parameters: {'n_estimators': 282, 'learning_rate': 0.021560865351963242, 'max_depth': 5, 'min_samples_leaf': 2, 'subsample': 0.7575838008479983}. Best is trial 0 with value: 1084.2619630578715.
[I 2026-02-12 09:39:14,771] Trial 1 finished with value: 1094.951336073134 and parameters: {'n_estimators': 345, 'learning_rate': 0.06201540323584707, 'max_depth': 4, 'min_samples_leaf': 15, 'subsample': 0.8451987753751687}. Best is trial 0 with value: 1084.2619630578715.
[I 2026-02-12 09:40:21,068] Trial 2 finished with value: 1087.1232827867188 and parameters: {'n_estimators': 538, 'learning_rate': 0.02677122351799263, 'max_depth': 3, 'min_samples_leaf': 2, 'subsample': 0.9500630979464961}. Best is trial 0 with value: 1084.2619630578715.
[I 2026-02-12 09:41:32,697] Trial 3 finished with value: 1115.3384

Best CV: 1077.3112550008311
Best Params: {'n_estimators': 497, 'learning_rate': 0.010630809583344102, 'max_depth': 3, 'min_samples_leaf': 9, 'subsample': 0.7220871744289936}


In [28]:
test_processed = test_processed[X.columns]


In [29]:
best_model = GradientBoostingRegressor(
    **study.best_params,
    random_state=RANDOM_STATE
)

best_model.fit(X, y)

predictions = best_model.predict(test_processed)

submission = test_ids.copy()
submission["Item_Outlet_Sales"] = predictions

submission.to_csv("submission_v4.csv", index=False)


In [30]:
with mlflow.start_run(run_name="Leaderboard_Log"):
    mlflow.log_metric("leaderboard_score", 1157)


In [31]:
mlflow.search_runs()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.leaderboard_score,metrics.cv_rmse,params.alpha,params.verbose,...,params.min_samples_leaf,params.tol,params.subsample,params.min_weight_fraction_leaf,params.random_state,params.max_depth,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.source.name
0,738c46b80d2142acb9d79227ac9318aa,1,FINISHED,/content/mlruns/1/738c46b80d2142acb9d79227ac93...,2026-02-12 10:30:37.542000+00:00,2026-02-12 10:30:37.574000+00:00,1157.0,NaN,None,None,...,None,None,None,None,None,None,root,Leaderboard_Log,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
1,db09e4a87cc243659bb5433bf03ea393,1,FINISHED,/content/mlruns/1/db09e4a87cc243659bb5433bf03e...,2026-02-12 10:27:44.438000+00:00,2026-02-12 10:29:07.368000+00:00,NaN,1095.310961,0.9,0,...,5,0.0001,0.8,0.0,42,4,root,GB_TargetEncoding_Baseline,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
2,976d3e4c9edb4166a8444ca16768292a,1,FINISHED,/content/mlruns/1/976d3e4c9edb4166a8444ca16768...,2026-02-12 10:21:14.470000+00:00,2026-02-12 10:22:16.803000+00:00,NaN,1097.376056,None,None,...,17,None,0.7094553919595316,None,42,3,root,caring-gnu-246,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
3,8654276f8abf411db468861a2a3c96a6,1,FINISHED,/content/mlruns/1/8654276f8abf411db468861a2a3c...,2026-02-12 10:19:54.049000+00:00,2026-02-12 10:21:14.456000+00:00,NaN,1088.971858,None,None,...,10,None,0.678200878039877,None,42,6,root,incongruous-moth-577,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
4,30bf4e018fc34ba9a662b84adb41b885,1,FINISHED,/content/mlruns/1/30bf4e018fc34ba9a662b84adb41...,2026-02-12 10:19:30.237000+00:00,2026-02-12 10:19:54.037000+00:00,NaN,1081.213742,None,None,...,19,None,0.6137426498367057,None,42,2,root,abundant-owl-486,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,c1808c8b49754a7594307344909a982e,1,FAILED,/content/mlruns/1/c1808c8b49754a7594307344909a...,2026-02-12 09:23:05.093000+00:00,2026-02-12 09:24:22.051000+00:00,NaN,NaN,None,None,...,None,None,None,None,None,None,root,unruly-duck-446,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
64,e444e3f1781d485a8c2ae88eccbac2f1,1,FINISHED,/content/mlruns/1/e444e3f1781d485a8c2ae88eccba...,2026-02-12 09:21:10.047000+00:00,2026-02-12 09:23:05.078000+00:00,NaN,1128.905671,None,None,...,6,None,0.8851227082768152,None,42,4,root,magnificent-loon-843,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
65,74b13ffa468c4ab88bb3911cae50bd2a,1,FINISHED,/content/mlruns/1/74b13ffa468c4ab88bb3911cae50...,2026-02-12 09:17:13.276000+00:00,2026-02-12 09:21:10.034000+00:00,NaN,1124.736978,None,None,...,5,None,0.9976689882043613,None,42,6,root,melodic-sheep-438,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
66,f840ac90fc3d4079a31a475db116f686,1,FINISHED,/content/mlruns/1/f840ac90fc3d4079a31a475db116...,2026-02-12 09:16:34.428000+00:00,2026-02-12 09:17:13.264000+00:00,NaN,1090.199922,None,None,...,7,None,0.8738891460796436,None,42,3,root,nimble-shad-706,NOTEBOOK,fileId=1Q-X-z9Ib62zn8oo4FzJW_X5KAHQAKTJ0
